# swmmx: all plotting functions

This notebook is a comprehensive reference for every public matplotlib plotting API in `swmmx`.

It covers whole-network layout plots, dynamic result time-series plots, longitudinal hydraulic profile plots, and every public input, default, output, variable family, save rule, and common error.

All plotting APIs use matplotlib directly and return the same core result: `(fig, ax)`.

In [ ]:
from pathlib import Path
import matplotlib
matplotlib.use('Agg')  # safe for notebooks, tests, and headless environments
import matplotlib.pyplot as plt
from swmmx import swmm

example_path = Path('examples/example.inp')
output_dir = Path('examples/output')
output_dir.mkdir(parents=True, exist_ok=True)
m = swmm(example_path)

# Public getters keep the examples model-aware instead of relying on placeholder IDs.
node_ids = list(m.get.node.id(format='np'))
link_ids = list(m.get.link.id(format='np'))
subcatchment_ids = list(m.get.subcatchment.id(format='np'))
rain_gage_ids = list(m.get.rain_gage.id(format='np'))
selected_node_ids = [node_ids[0], node_ids[-1]] if len(node_ids) >= 2 else node_ids[:1]
selected_link_ids = link_ids[:2]
selected_subcatchment_ids = subcatchment_ids[:2]
selected_rain_gage_ids = rain_gage_ids[:1]
node_class_data = {node_id: ('Outfall' if node_id.lower() == 'outlet' else 'Junction') for node_id in selected_node_ids}
node_size_data = {node_id: index + 1 for index, node_id in enumerate(selected_node_ids)}
link_width_data = {link_id: index + 1 for index, link_id in enumerate(selected_link_ids)}
link_class_data = {link_id: ('Main' if index == 0 else 'Secondary') for index, link_id in enumerate(selected_link_ids)}
subcatchment_class_data = {subcatchment_id: ('Low' if index == 0 else 'High') for index, subcatchment_id in enumerate(selected_subcatchment_ids)}


## Plotting families

| Public API | Purpose | Needs results? | Return value |
| --- | --- | --- | --- |
| `m.plot_layout(...)` | Draw mapped nodes, links, subcatchments, rain gages, labels, and optional data-driven styling. | Only for result-driven styling. | `(fig, ax)` |
| `m.plot_timeseries.<category>.<variable>(ids=None, ...)` | Plot one or many result series against time. | Yes. | `(fig, ax)` |
| `m.plot_profile.nodes(...)` / `.links(...)` / `.longest(...)` | Plot a longitudinal hydraulic path with geometry and optional result overlays. | Only for HGL/water overlays. | `(fig, ax)` |

## Shared save behavior

| Case | Behavior |
| --- | --- |
| No `save_path`, no `save_format` | Do not save. |
| Only `save_format` | Save beside the model file, or in the current working directory for unsaved models. |
| Existing folder in `save_path` | Create a sensible filename inside that folder. |
| Path without extension | Append the requested format, or `.png` by default. |
| Path with extension | Infer format from extension unless `save_format` overrides it. |
| Supported formats | `png`, `jpg`, `jpeg`, `pdf`, `svg`, `tiff`. |

## `m.plot_layout()`

Layout plots read map geometry from `[COORDINATES]`, `[VERTICES]`, `[POLYGONS]`, and `[SYMBOLS]`. Elements with unusable coordinates are skipped with warnings; if no plottable geometry exists, the call raises `PlotDataError`. Subcatchment centroids are derived from polygon geometry, then connected to outlet nodes or downstream subcatchments with dashed lines. Node/link symbology is type-aware by default, and LID usage markers are drawn at subcatchment centroids when present. Titles, grids, and visible coordinate axes use safe ordinary artists instead of Matplotlib's native title/tick/grid machinery, avoiding the Spyder/Agg recursive tick-copy path. On non-interactive matplotlib canvases such as Agg, `show=True` avoids useless GUI calls but keeps the figure available for Spyder/Jupyter rendering; `show=False` removes only the library-created figure manager so hidden plots stay hidden.

| Input | Type | Default | Meaning |
| --- | --- | --- | --- |
| `legend` | `bool` | `True` | Show ordinary layer legend entries. |
| `grid` | `bool` | `False` | Show reference grid lines; they remain visible even when coordinate labels stay hidden. |
| `title` | `str \| None` | `None` | Optional plot title. |
| `legend_title` | `str \| None` | `None` | Optional legend title. |
| `axis` | `bool` | `False` | Show coordinate axes and ticks; hidden by default for maps. |
| `x_axis_title` | `str \| None` | `None` | Optional x-axis title when `axis=True`. |
| `y_axis_title` | `str \| None` | `None` | Optional y-axis title when `axis=True`. |
| `save_format` | `str \| None` | `None` | Optional save format: png, jpg, jpeg, pdf, svg, or tiff. |
| `save_path` | `str \| Path \| None` | `None` | Optional file path or existing folder target. |
| `figsize` | `tuple[float, float]` | `(10, 8)` | Figure size when `ax` is not supplied. |
| `dpi` | `int` | `300` | Figure resolution for new figures and saved output. |
| `ax` | matplotlib `Axes \| None` | `None` | Draw into existing axes when supplied. |
| `show` | `bool` | `True` | Display the completed figure; `False` suppresses automatic notebook display for function-created figures. |
| `nodes` | `dict \| bool \| None` | `None` | Node-layer styling dictionary or visibility flag. |
| `links` | `dict \| bool \| None` | `None` | Link-layer styling dictionary or visibility flag. |
| `subcatchments` | `dict \| bool \| None` | `None` | Subcatchment-layer styling dictionary or visibility flag. |
| `subcatchment_outlets` | `dict \| bool \| None` | `None` | Dashed subcatchment-to-outlet connector layer. |
| `rain_gages` | `dict \| bool \| None` | `None` | Rain-gage-layer styling dictionary or visibility flag. |
| `lids` | `dict \| bool \| None` | `None` | LID-usage marker layer. |
| `labels` | `dict \| bool \| None` | `None` | Label-layer styling dictionary or visibility flag. |
| `link_color_by` | `str \| None` | `None` | Alias for parameter-driven link color. |
| `link_color_mode` | `str \| None` | `None` | Alias mode for link color: continuous or discrete. |
| `link_cmap` | `str \| None` | `None` | Alias colormap for link styling. |
| `node_color_result` | `str \| None` | `None` | Alias for result-driven node color variable. |
| `node_result_aggregation` | `str \| None` | `None` | Alias aggregation for node result styling. |
| `link_user_data` | `mapping \| Series \| None` | `None` | Alias for user-driven link color data. |

### Layout layer dictionaries

| Layer | Supported keys | Important defaults |
| --- | --- | --- |
| `nodes` | `visible`, `label`, `legend`, `alpha`, `zorder`, `size`, `color`, `edge_color`, `marker`, `type_markers`, `linewidth`, `ids` | Type markers: junction `o`, outfall `v`, divider `D`, storage `s` |
| `links` | `visible`, `label`, `legend`, `alpha`, `zorder`, `width`, `size`, `color`, `linestyle`, `type_linestyles`, `ids` | Type lines: conduit solid, pump dash-dot, orifice dotted, weir dashed, outlet long-dashed |
| `subcatchments` | `visible`, `label`, `legend`, `alpha`, `zorder`, `color`, `edge_color`, `linewidth`, `ids` | `color='lightgreen'`, `edge_color='green'`, `alpha=0.25` |
| `subcatchment_outlets` | `visible`, `label`, `legend`, `alpha`, `zorder`, `width`, `color`, `linestyle`, `ids` | Dashed gray connector from centroid to outlet |
| `rain_gages` | `visible`, `label`, `legend`, `alpha`, `zorder`, `size`, `color`, `marker`, `ids` | `size=45`, `color='tab:blue'`, `marker='^'` |
| `lids` | `visible`, `label`, `legend`, `alpha`, `zorder`, `size`, `color`, `edge_color`, `linewidth`, `markers` | One distinct marker per LID control ID when usages exist |
| `labels` | `visible`, `alpha`, `zorder`, `fontsize`, `color` | `visible=False`, `fontsize=8`, `color='black'` |

### Data-driven layout style dictionaries

The `color`, node `size`, and link `width`/`size` keys can be static values or richer dictionaries. Result-driven encodings require `m.run()` first.

| Key | Accepted values | Meaning |
| --- | --- | --- |
| `by` | `static`, `parameter`, `result`, or `user` | Select where styling values come from. |
| `value` | static color/size/width | Used when `by='static'`. |
| `category`, `variable` | public getter names | Required for `parameter` and `result` styling. |
| `data` | mapping or pandas Series | Required for `user` styling. |
| `mode` | `continuous` or `discrete` | Color mapping mode; sizes/widths currently scale numeric values continuously. |
| `cmap` | matplotlib colormap name | Colormap for continuous or discrete color encoding. |
| `vmin`, `vmax` | numeric bounds | Optional continuous color scaling bounds. |
| `aggregation` | `last`, `max`, `min`, `mean`, `median`, or `sum` | Collapse result time series to one value per object. |
| `time_step`, `time` | integer index or timestamp-like value | Select one result instant instead of aggregating. |
| `legend`, `legend_title` | `bool`, `str` | Control the dedicated custom-style legend section. |
| `min_size`, `max_size` | numeric bounds | Node-size scaling bounds. |
| `min_width`, `max_width` | numeric bounds | Link-width scaling bounds. |
| `labels`, `bins` | sequence metadata | Accepted style metadata for discrete presentation; labels are used for color categories. |

### Layout examples

```python
m.plot_layout()

m.plot_layout(
    nodes={'size': 40, 'color': 'black'},
    links={'width': 1.5, 'color': 'gray'},
)

m.plot_layout(
    links={
        'color': {
            'by': 'parameter',
            'category': 'conduit',
            'variable': 'roughness',
            'mode': 'continuous',
            'cmap': 'viridis',
        }
    }
)

m.plot_layout(
    nodes={'ids': selected_node_ids, 'size': {'by': 'user', 'data': node_size_data}},
    links={'ids': selected_link_ids, 'width': {'by': 'user', 'data': link_width_data}},
    subcatchments={'ids': selected_subcatchment_ids, 'color': {'by': 'parameter', 'variable': 'area', 'mode': 'continuous'}},
)

m.run()
m.plot_layout(
    nodes={'size': {'by': 'result', 'variable': 'depth', 'aggregation': 'max'}},
    links={'color': {'by': 'result', 'category': 'link', 'variable': 'flow', 'aggregation': 'max'}},
)

m.plot_layout(link_color_by='roughness', link_color_mode='continuous', link_cmap='viridis')
```


### Seven fully annotated `m.plot_layout()` layer examples

These examples use IDs discovered from `examples/example.inp`, so they are safe to execute after the setup cell above.  
Result-driven examples require `m.run()` first; user-driven examples use dictionaries prepared in the setup cell.

#### 1. `nodes={...}`

```python
m.plot_layout(
    nodes={
        'visible': True,             # True draws nodes; False hides the whole node layer.
        'label': 'Nodes',             # Generic layer label used in the ordinary object legend.
        'legend': True,               # Include node information in legends when True.
        'alpha': 1.0,                 # Marker transparency from 0.0 (clear) to 1.0 (opaque).
        'zorder': 3,                   # Drawing order; larger numbers appear above lower layers.
        'size': {
            'by': 'user',             # Choose 'static', 'parameter', 'result', or 'user'.
            # 'value': 40,            # Static size, used only when by='static'.
            # 'category': 'node',     # Public getter category for parameter/result styling.
            # 'variable': 'depth',    # Public getter variable to encode.
            'data': node_size_data,   # ID -> value mapping, used only when by='user'.
            'mode': 'continuous',     # Reserved metadata for size encodings; numeric scaling is continuous.
            # 'bins': [0, 1, 2],      # Optional metadata for future/discrete presentation.
            # 'labels': ['low', 'high'],  # Optional presentation labels.
            # 'aggregation': 'max',   # Result reducer: last/max/min/mean/median/sum.
            # 'time_step': -1,        # Select one result row instead of aggregating.
            # 'time': '2026-01-01 01:00',  # Select one result timestamp instead of aggregating.
            'min_size': 20,            # Smallest rendered marker area.
            'max_size': 200,           # Largest rendered marker area.
            'legend': True,            # Add a dedicated custom-style legend section.
            'legend_title': 'Node size score',  # Custom legend section title.
        },
        'color': {
            'by': 'user',             # Here color is based on user-defined classes.
            # 'value': 'black',       # Static color, used only when by='static'.
            # 'category': 'node',     # Required for parameter/result styling.
            # 'variable': 'depth',    # Required for parameter/result styling.
            'data': node_class_data,  # ID -> value mapping, required when by='user'.
            'mode': 'discrete',        # 'continuous' for numeric ramps, 'discrete' for classes.
            'cmap': 'tab10',           # Any matplotlib colormap name.
            # 'bins': [0, 1, 2],      # Optional metadata for custom/discrete grouping.
            'labels': ['Junction', 'Outfall'],  # Display labels for discrete classes.
            # 'vmin': 0.0,            # Lower color bound for continuous color maps.
            # 'vmax': 2.0,            # Upper color bound for continuous color maps.
            # 'aggregation': 'max',   # Result reducer when by='result'.
            # 'time_step': -1,        # Single result row when by='result'.
            # 'time': '2026-01-01 01:00',  # Single result timestamp when by='result'.
            'legend': True,            # Add a dedicated color legend section.
            'legend_title': 'Node class',
        },
        'edge_color': 'white',         # Marker edge color.
        'marker': 'o',                 # Fallback marker when no type-specific marker is supplied.
        'type_markers': {              # Marker by SWMM node type.
            'junction': 'o',
            'outfall': 'v',
            'divider': 'D',
            'storage_unit': 's',
        },
        'linewidth': 0.5,              # Marker edge-line width.
        'ids': selected_node_ids,       # Optional node subset; omit to draw all nodes.
    },
)
```

#### 2. `links={...}`

```python
m.plot_layout(
    links={
        'visible': True,               # True draws links; False hides the whole link layer.
        'label': 'Links',               # Generic layer label for the ordinary object legend.
        'legend': True,                 # Include link information in legends when True.
        'alpha': 1.0,                   # Line transparency.
        'zorder': 2,                     # Draw links above polygons but below nodes.
        'width': {
            'by': 'parameter',          # Width can come from static/parameter/result/user data.
            # 'value': 1.5,             # Static width when by='static'.
            'category': 'conduit',      # Getter category for the width source.
            'variable': 'roughness',    # Getter variable for the width source.
            # 'data': link_width_data,  # ID -> value mapping when by='user'.
            'mode': 'continuous',       # Numeric width scaling is continuous.
            # 'bins': [0.01, 0.02],     # Optional metadata.
            # 'labels': ['smooth', 'rough'],  # Optional presentation labels.
            # 'aggregation': 'max',      # Result reducer when by='result'.
            # 'time_step': -1,           # Single result row when by='result'.
            # 'time': '2026-01-01 01:00',  # Single result timestamp when by='result'.
            'min_width': 0.5,            # Smallest rendered line width.
            'max_width': 4.0,            # Largest rendered line width.
            'legend': True,              # Add a dedicated width legend section.
            'legend_title': 'Conduit roughness',
        },
        # 'size': {...},                # Alias for 'width'; use either 'width' or 'size'.
        'color': {
            'by': 'user',               # Here color is based on your own link classes.
            # 'value': 'gray',          # Static color when by='static'.
            # 'category': 'link',       # Result getter category.
            # 'variable': 'flow',       # Result getter variable.
            'data': link_class_data,    # ID -> value mapping when by='user'.
            'mode': 'discrete',          # Discrete classes or continuous numeric values.
            'cmap': 'tab10',             # Matplotlib colormap.
            # 'bins': [0, 1, 2],        # Optional metadata.
            'labels': ['Main', 'Secondary'],  # Optional labels for discrete mode.
            # 'vmin': 0.0,              # Lower color-map limit for continuous mode.
            # 'vmax': 10.0,              # Optional upper color-map limit.
            # 'aggregation': 'max',      # Result reducer.
            # 'time_step': -1,           # Single result row instead of aggregation.
            # 'time': '2026-01-01 01:00',  # Single result timestamp instead of aggregation.
            'legend': True,              # Add a result-driven custom legend section.
            'legend_title': 'Link class',
        },
        'linestyle': '-',               # Fallback line style.
        'type_linestyles': {            # Line style by SWMM link type.
            'conduit': '-',
            'pump': '-.',
            'orifice': ':',
            'weir': '--',
            'outlet': (0, (5, 1)),
        },
        'ids': selected_link_ids,        # Optional link subset; omit to draw all links.
    },
)
```

#### 3. `subcatchments={...}`

```python
m.plot_layout(
    subcatchments={
        'visible': True,                # Draw polygon areas when True.
        'label': 'Subcatchments',        # Ordinary legend label.
        'legend': True,                  # Include polygons in legends when True.
        'alpha': 0.25,                   # Polygon fill transparency.
        'zorder': 1,                      # Background layer below network geometry.
        'color': {
            'by': 'parameter',           # Static/parameter/result/user are all supported.
            # 'value': 'lightgreen',     # Static fill color when by='static'.
            'category': 'subcatchment',  # Getter category for parameter/result styling.
            'variable': 'area',          # Getter variable to encode.
            # 'data': subcatchment_class_data,  # ID -> value mapping when by='user'.
            'mode': 'continuous',        # Continuous or discrete color mapping.
            'cmap': 'YlGn',               # Matplotlib colormap.
            # 'bins': [0, 1, 2],         # Optional metadata.
            # 'labels': ['small', 'large'],  # Optional labels for discrete mode.
            # 'vmin': 0.0,               # Optional lower color bound.
            # 'vmax': 10.0,              # Optional upper color bound.
            # 'aggregation': 'max',      # Result reducer when by='result'.
            # 'time_step': -1,           # Single result row when by='result'.
            # 'time': '2026-01-01 01:00',  # Single result timestamp when by='result'.
            'legend': True,              # Add a custom legend section for the fill encoding.
            'legend_title': 'Subcatchment area',
        },
        'edge_color': 'green',           # Polygon boundary color.
        'linewidth': 1.0,                # Polygon boundary width.
        'ids': selected_subcatchment_ids, # Optional polygon subset.
    },
)
```

#### 4. `subcatchment_outlets={...}`

```python
m.plot_layout(
    subcatchment_outlets={
        'visible': True,                 # Draw centroid -> outlet connectors.
        'label': 'Subcatchment outlets',  # Ordinary legend label.
        'legend': True,                   # Include connector style in the legend.
        'alpha': 0.8,                     # Connector transparency.
        'zorder': 1.5,                    # Above polygons, below links.
        'width': 1.0,                     # Connector line width.
        'color': '0.45',                  # Any matplotlib color.
        'linestyle': '--',                # Dashed drainage cue.
        'ids': selected_subcatchment_ids, # Optional subcatchment subset.
    },
)
```

#### 5. `rain_gages={...}`

```python
m.plot_layout(
    rain_gages={
        'visible': True,                  # Draw rain-gage symbols when True.
        'label': 'Rain gages',             # Ordinary legend label.
        'legend': True,                    # Include rain gages in the legend.
        'alpha': 1.0,                      # Marker transparency.
        'zorder': 4,                       # Above the hydraulic network.
        'size': 45,                        # Marker area.
        'color': 'tab:blue',               # Marker fill color.
        'marker': '^',                     # Matplotlib marker shape.
        'ids': selected_rain_gage_ids,     # Optional rain-gage subset.
    },
)
```

#### 6. `lids={...}`

```python
m.plot_layout(
    lids={
        'visible': True,                   # Draw LID usage markers when usages exist.
        'label': 'LID controls',            # Generic layer label; each control also gets its own legend item.
        'legend': True,                     # Include LID markers in the legend.
        'alpha': 1.0,                       # Marker transparency.
        'zorder': 4.5,                      # Above nodes/rain gages when desired.
        'size': 55,                         # Marker area.
        'color': 'tab:purple',              # Marker fill color.
        'edge_color': 'white',              # Marker edge color.
        'linewidth': 0.5,                   # Marker edge width.
        'markers': ['P', 'X', '*', 'h', '8', 'd'],  # Cycled across distinct LID control IDs.
    },
)
```

#### 7. `labels={...}`

```python
m.plot_layout(
    labels={
        'visible': True,                    # Draw node-ID text labels when True.
        'alpha': 1.0,                       # Text transparency.
        'zorder': 5,                         # Text usually sits above all symbol layers.
        'fontsize': 8,                       # Text size in points.
        'color': 'black',                    # Text color.
    },
)
```

In [ ]:
fig, ax = m.plot_layout(
    title='Network layout',
    nodes={'size': 40, 'color': 'black'},
    links={'width': 1.5, 'color': 'gray'},
    show=False,
    save_path=output_dir / 'notebook_layout_example.png',
)
plt.close(fig)


## `m.plot_timeseries.<category>.<variable>()`

Time-series endpoints are generated dynamically from public result variables. They require a completed run and use a timestamp index by default. Their titles, grids, ticks, and axis labels use safe ordinary artists instead of native Matplotlib `Axis` artists, while legends use lightweight proxy lines rather than cloning the plotted artists. Together those two choices avoid the Spyder/Agg recursive marker-copy paths.

| Input | Type | Default | Meaning |
| --- | --- | --- | --- |
| `ids` | `None \| str \| list[str]` | `None` | Select all, one, or several objects where the category is object-based. |
| `legend` | `bool` | `True` | Show line legend. |
| `grid` | `bool` | `True` | Show grid lines. |
| `title` | `str \| None` | `None` | Optional title; otherwise generated from category and variable. |
| `legend_title` | `str \| None` | `None` | Optional legend title. |
| `axis` | `bool` | `True` | Show axes and ticks. |
| `x_axis_title` | `str \| None` | `None` | Optional x-axis title; defaults from the selected time format. |
| `y_axis_title` | `str \| None` | `None` | Optional y-axis title; defaults to the variable name. |
| `save_format` | `str \| None` | `None` | Optional save format. |
| `save_path` | `str \| Path \| None` | `None` | Optional file path or existing folder target. |
| `figsize` | `tuple[float, float]` | `(10, 4)` | Figure size when `ax` is not supplied. |
| `dpi` | `int` | `300` | Figure resolution. |
| `ax` | matplotlib `Axes \| None` | `None` | Compose into existing axes. |
| `show` | `bool` | `True` | Call `plt.show()` after rendering. |
| `time_format` | `'datetime' \| 'clock' \| 'elapsed'` | `'datetime'` | Use full date-time labels, hour-minute labels, or elapsed hours. |
| `start_time`, `end_time` | timestamp-like | `None` | Optional time-window filters. |
| `labels` | `dict \| list \| tuple \| None` | `None` | Custom line labels by column name or plot order. |
| `linewidth` | `float` | `1.5` | Line width. |
| `linestyle` | `str` | `'-'` | Matplotlib line style. |
| `marker` | matplotlib marker or `None` | `None` | Optional point marker. |
| `alpha` | `float` | `1.0` | Line transparency. |

### Available time-series variables

| Category | Available variables | ID behavior | Example endpoint |
| --- | --- | --- | --- |
| `conduit` | `capacity`, `depth`, `flow`, `velocity` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.conduit.capacity(...)` |
| `inlet` | `captured_flow` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.inlet.captured_flow(...)` |
| `lid_usage` | `drain_outflow`, `evaporation`, `infiltration`, `inflow`, `storage`, `surface_outflow` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.lid_usage.drain_outflow(...)` |
| `link` | `capacity`, `depth`, `flow`, `pollutant_concentration`, `setting`, `velocity`, `volume` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.link.capacity(...)` |
| `node` | `depth`, `flooding`, `head`, `lateral_inflow`, `overflow`, `pollutant_concentration`, `total_inflow`, `volume` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.node.depth(...)` |
| `orifice` | `flow`, `setting` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.orifice.flow(...)` |
| `outlet` | `flow`, `setting` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.outlet.flow(...)` |
| `pump` | `energy`, `flow`, `setting`, `status` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.pump.energy(...)` |
| `rain_gage` | `rainfall` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.rain_gage.rainfall(...)` |
| `street` | `spread` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.street.spread(...)` |
| `subcatchment` | `evaporation`, `groundwater_elevation`, `groundwater_flow`, `infiltration`, `pollutant_concentration`, `rainfall`, `runoff`, `snow_depth`, `soil_moisture` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.subcatchment.evaporation(...)` |
| `system` | `air_temperature`, `direct_inflow`, `dry_weather_inflow`, `evaporation`, `evaporation_loss`, `flooding`, `groundwater_inflow`, `infiltration`, `outflow`, `rainfall`, `rdii_inflow`, `runoff`, `snow_depth`, `total_lateral_inflow`, `volume` | System-wide series; `ids` is not needed | `m.plot_timeseries.system.air_temperature(...)` |
| `system_result` | `air_temperature`, `direct_inflow`, `dry_weather_inflow`, `evaporation`, `flooding`, `groundwater_inflow`, `infiltration`, `outfall_flow`, `pollutant_loading`, `rainfall`, `rdii_inflow`, `runoff`, `snow_depth`, `storage_volume`, `total_lateral_inflow` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.system_result.air_temperature(...)` |
| `weir` | `flow`, `setting` | `ids=None`, one ID string, or a list of ID strings | `m.plot_timeseries.weir.flow(...)` |

### Time-series examples

```python
m.run()
m.plot_timeseries.link.flow(['C1', 'C2'])
m.plot_timeseries.node.depth('J1', title='Node depth')
m.plot_timeseries.link.flow('C1', time_format='clock')
m.plot_timeseries.system.runoff(time_format='elapsed')
```


In [ ]:
# Result-based plotting examples need an executed model.
m.run()
fig, ax = m.plot_timeseries.link.flow(
    ids=['P001', 'P005'],
    title='Conduit flow',
    y_axis_title='Flow',
    show=False,
    save_path=output_dir / 'notebook_timeseries_example.png',
)
plt.close(fig)


## `m.plot_profile`

| Public API | Path selector input | Behavior |
| --- | --- | --- |
| `m.plot_profile.nodes(start_node, end_node, ...)` | Existing start/end node IDs | Find a directed hydraulic path between nodes, then plot it. |
| `m.plot_profile.links(ids, ...)` | One ordered link ID or list of link IDs | Validate connected order in either direction, then plot exactly that sequence. |
| `m.plot_profile.longest(...)` | No path selector | Find the longest directed conduit path and plot it. |

### Profile inputs

Profiles use directed conduit connectivity, node invert elevations, conduit lengths, and conduit full depths. Ground elevation is approximated from node invert plus max depth when needed. EGL is currently not exposed by the result layer, so requesting `show_egl=True` emits a warning and skips that overlay.

| Input | Type | Default | Meaning |
| --- | --- | --- | --- |
| `time_step` | `int` | `-1` | Result row index for overlay variables; `-1` means last row. |
| `aggregation` | `str \| None` | `None` | Optional result aggregation: last, max, min, mean, or median. |
| `legend` | `bool` | `True` | Show profile legend. |
| `grid` | `bool` | `True` | Show grid lines. |
| `title` | `str \| None` | `None` | Optional profile title. |
| `legend_title` | `str \| None` | `None` | Optional legend title. |
| `axis` | `bool` | `True` | Show axes and ticks. |
| `x_axis_title` | `str \| None` | `None` | Optional x-axis title; defaults to Distance. |
| `y_axis_title` | `str \| None` | `None` | Optional y-axis title; defaults to Elevation. |
| `save_format` | `str \| None` | `None` | Optional save format. |
| `save_path` | `str \| Path \| None` | `None` | Optional file path or existing folder target. |
| `figsize` | `tuple[float, float]` | `(12, 5)` | Figure size when `ax` is not supplied. |
| `dpi` | `int` | `300` | Figure resolution. |
| `ax` | matplotlib `Axes \| None` | `None` | Compose into existing axes. |
| `show` | `bool` | `True` | Call `plt.show()` after rendering. |
| `unit_length`, `unit_elevation` | `str \| None` | `None` | Optional axis-unit overrides; otherwise inferred from the model flow-unit family. |
| `show_ground` | `bool` | `True` | Draw approximated ground line. |
| `show_conduits` | `bool` | `True` | Draw conduit barrels. |
| `show_invert` | `bool` | `True` | Draw node invert line. |
| `show_crown` | `bool` | `True` | Draw conduit crown line. |
| `show_hgl` | `bool` | `False` | Draw hydraulic grade line; requires results. |
| `show_egl` | `bool` | `False` | Request energy grade line; currently warns and skips because EGL is not exposed. |
| `show_water_depth` | `bool` | `False` | Draw water level; requires results. |
| `show_node_labels` | `bool` | `True` | Annotate nodes. |
| `show_link_labels` | `bool` | `True` | Annotate links on their conduit barrels. |
| `show_node_guides` | `bool` | `True` | Draw vertical dotted lines at node locations. |
| `show_surcharge` | `bool` | `True` | Mark surcharge locations when results are available. |
| `show_flooding` | `bool` | `True` | Mark flooding locations when results are available. |
| `fill_conduits` | `bool` | `True` | Shade conduit barrels between invert and crown. |
| `line_styles` | `dict[str, str] \| None` | `None` | Override styles for `ground`, `invert`, `crown`, `hgl`, or `water`. |
| `colors` | `dict[str, str] \| None` | `None` | Override colors for `ground`, `invert`, `crown`, `conduit`, `hgl`, `water`, `flooding`, or `surcharge`. |

### Profile examples

```python
m.plot_profile.nodes('J1', 'OUT1', show_hgl=True, aggregation='max')
m.plot_profile.links(['C1', 'C2', 'C3'])  # connected walks may follow or oppose hydraulic direction
m.plot_profile.longest(show_hgl=True, aggregation='max')
```


In [ ]:
fig, ax = m.plot_profile.longest(
    show_hgl=True,
    aggregation='max',
    title='Longest path profile',
    show=False,
    save_path=output_dir / 'notebook_profile_example.png',
)
plt.close(fig)


## Validation and common errors

| Error | Typical cause |
| --- | --- |
| `PlotDataError` | Missing coordinates, invalid style data, non-time-series variable, invalid profile geometry, or invalid time selection. |
| `ModelNotRunError` | Result-driven layout styling, time-series plotting, or requested profile result overlays before `m.run()`. |
| `UnknownIDError` | Requested node/link/object ID is not present. |
| `UnknownCategoryError` / `UnknownParameterError` | Unknown dynamic time-series category or variable. |
| `NoPathError` | No directed hydraulic path exists between requested nodes. |
| `InvalidPathError` | Supplied profile links are empty or not connected in order. |
| `SaveError` | Unsupported save format or a figure cannot be written. |